In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
%pwd

'/projectnb/batmanlab/mragoza/lung-project/notebooks/bmc-4dct'

In [5]:
import sys, os

print(os.getpid())

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

def add_path(p: str):
    if p not in sys.path:
        sys.path.append(p)

add_path(os.environ['LP_ROOT'])
add_path(os.environ['PROJECT'])

import project

import torch
torch.cuda.is_available()

506380


False

# Gather examples

In [7]:
%autoreload
from pathlib import Path

data_root = Path(os.environ['PRIVATE']) / 'data' / 'BMC-4DCT'
assert data_root.exists()

#subject_file = data_root / 'sample1000_2025-07-22.csv'
#subject_list = project.fileio.load_subject_list(subject_file, key='sid', sep='\t')

subject_list = ['4DCT_1_1']

data_config = {
    'name': 'BMC-4DCT',
    'root': str(data_root),
    'subjects': subject_list,
    'state_pairs': [(50, 0)],
    'variant': 'TEST',
    'pipeline_tags': {}
}

examples = project.get_examples(data_config)

Gathering examples...
1 total


In [8]:
ex = examples[0]
project.utils.pprint(ex)

Example()
├── dataset:  'BMC-4DCT'
├── subject:  '4DCT_1_1'
├── variant:  'TEST'
├── paths:    dict(len=10)
|   ├── 'ref_state':       dict(len=1)
|   ├── 'init_state':      dict(len=4)
|   ├── 'curr_state':      dict(len=4)
|   ├── 'anatomical_map':  PosixPath('/restricted/projectnb/batmanlab/mragoza/data/BMC-4DCT/processed/TEST/4DCT_1_1/masks/4DCT_1_1_50_std_seg_anat.nii.gz')
|   ├── 'anatomical_mesh': PosixPath('/restricted/projectnb/batmanlab/mragoza/data/BMC-4DCT/processed/TEST/4DCT_1_1/meshes/4DCT_1_1_50_std_seg_anat_gen.xdmf')
|   ├── 'disp_field':      PosixPath('/restricted/projectnb/batmanlab/mragoza/data/BMC-4DCT/processed/TEST/4DCT_1_1/fields/4DCT_1_1_50_std_seg_reg_0.nii.gz')
|   ├── 'interp_mesh':     PosixPath('/restricted/projectnb/batmanlab/mragoza/data/BMC-4DCT/processed/TEST/4DCT_1_1/meshes/4DCT_1_1_50_std_seg_reg_0_anat_gen_int.xdmf')
|   ├── 'input_image':     PosixPath('/restricted/projectnb/batmanlab/mragoza/data/BMC-4DCT/processed/TEST/4DCT_1_1/images/4DCT_1_1_5

In [9]:
project.fileio.load_config('../copdgene/2026-09-15_config.yaml')

Loading ../copdgene/2026-09-15_config.yaml


{'dataset': {'name': 'COPDGene',
  'root': '/restricted/projectnb/batmanlab/mragoza/data/COPDGene',
  'subjects': '/restricted/projectnb/batmanlab/mragoza/data/COPDGene/sample1000_2025-07-22.csv',
  'state_pairs': [['EXP', 'INSP']],
  'variant': '2026-09-15',
  'pipeline_tags': {'image_resampling': 'iso',
   'image_segmentation': 'v3d',
   'image_registration': 'gil',
   'anatomical_regions': 'lung',
   'material_properties': 'bela',
   'mesh_generation': 'rad',
   'field_interpolation': 'fix',
   'forward_simulation': 'LE'}},
 'preprocessing': {'image_resampling': {'spacing': [1.0, 1.0, 1.0],
   'interpolator': 'bspline',
   'default_value': -1000.0},
  'image_segmentation': {'tasks': [{'method': 'TotalSegmentator',
     'kwargs': {'task': 'total',
      'roi_subset': ['lung_upper_lobe_right',
       'lung_middle_lobe_right',
       'lung_lower_lobe_right',
       'lung_upper_lobe_left',
       'lung_lower_lobe_left']}},
    {'method': 'TotalSegmentator', 'kwargs': {'task': 'lung_vess

In [12]:
LUNG_ROIS = ['lung_upper_lobe_right', 'lung_middle_lobe_right', 'lung_lower_lobe_right', 'lung_upper_lobe_left', 'lung_lower_lobe_left']

proc_config = {
    'image_resampling': {'spacing': [1.0, 1.0, 1.0], 'interpolator': 'bspline', 'default_value': -1000.},
    'image_segmentation': {
        'tasks': [
            {
                'method': 'TotalSegmentator',
                'kwargs': {'task': 'total', 'roi_subset': LUNG_ROIS}
            },
            {
                'method': 'TotalSegmentator',
                'kwargs': {'task': 'lung_vessels'}
            },
            {
                'method': 'VisionFeature',
                'kwargs': {'model': 'vista3d', 'results_dir': '/restricted/projectnb/batmanlab/haozhe/Lung_Mesh_Segment/data/nnUNetv2/nnUNet_results'}
            },
            {
                'method': 'HU_threshold', 
                'kwargs': {
                    'thresholds': {
                        'lt-950': {'operator': '<', 'value': -950},
                        'lt-910': {'operator': '<', 'value': -910},
                        'lt-850': {'operator': '<', 'value': -850},
                        'gt-500': {'operator': '>', 'value': -500},
                        'gt-200': {'operator': '>', 'value': -200},
                    }
                }
            }
        ]
    },
    'image_registration': {
        'methods': 'unigradicon',
        'kwargs': {
            'model': 'gradiconlung',
            'similarity': 'lncc2',
            'sigma': 1,
            'iterations': 50,
            'loss_function_masking': True,
            'intensity_conservation_loss': True,
            'weights_root': '/projectnb/batmanlab/mragoza/lung-project/network_weights'
        }
    },
    'anatomical_regions': {
        'roi_order': LUNG_ROIS,
        'filter_kws': {'min_voxels': 30, 'keep_largest': True}
    },
    'field_interpolation': {'displacement_key': 'u_reg'}
}
proc_config

{'image_resampling': {'spacing': [1.0, 1.0, 1.0],
  'interpolator': 'bspline',
  'default_value': -1000.0},
 'image_segmentation': {'tasks': [{'method': 'TotalSegmentator',
    'kwargs': {'task': 'total',
     'roi_subset': ['lung_upper_lobe_right',
      'lung_middle_lobe_right',
      'lung_lower_lobe_right',
      'lung_upper_lobe_left',
      'lung_lower_lobe_left']}},
   {'method': 'TotalSegmentator', 'kwargs': {'task': 'lung_vessels'}},
   {'method': 'VisionFeature',
    'kwargs': {'model': 'vista3d',
     'results_dir': '/restricted/projectnb/batmanlab/haozhe/Lung_Mesh_Segment/data/nnUNetv2/nnUNet_results'}},
   {'method': 'HU_threshold',
    'kwargs': {'thresholds': {'lt-950': {'operator': '<', 'value': -950},
      'lt-910': {'operator': '<', 'value': -910},
      'lt-850': {'operator': '<', 'value': -850},
      'gt-500': {'operator': '>', 'value': -500},
      'gt-200': {'operator': '>', 'value': -200}}}}]},
 'image_registration': {'methods': 'unigradicon',
  'kwargs': {'mod

In [13]:
import project.preprocessing
project.preprocessing.preprocess_example(ex, proc_config)

INFO: Skipping stage resample_image_spacing
INFO: Running stage create_segmentation_masks
Running TotalSegmentator task: 'total'
No GPU detected. Running on CPU. This can be very slow. The '--fast' or the `--roi_subset` option can help to reduce runtime.

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
Resampling...
  Resampled in 0.09s
Predicting...


100%|██████████| 4/4 [00:03<00:00,  1.05it/s]


  Predicted in 33.39s
Resampling...
  Resampled in 2.71s
  cropping from (500, 500, 349) to (296, 241, 247)
Resampling...
  Resampled in 0.66s
Predicting part 1 of 1 ...


100%|██████████| 8/8 [00:41<00:00,  5.24s/it]


  Predicted in 62.25s
Resampling...
  Resampled in 0.48s
Saving segmentations...
  Saved in 5.12s
Running TotalSegmentator task: 'lung_vessels'
No GPU detected. Running on CPU. This can be very slow. The '--fast' or the `--roi_subset` option can help to reduce runtime.

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
  (Using more robust (but slower) 3mm model for cropping.)
Resampling...
  Resampled in 0.45s
Predicting...


100%|██████████| 8/8 [00:27<00:00,  3.42s/it]


  Predicted in 51.07s
Resampling...
  Resampled in 1.67s
  cropping from (500, 500, 349) to (296, 236, 248)
Resampling...
  Resampled in 2.64s
Predicting...


100%|██████████| 100/100 [08:02<00:00,  4.83s/it]


  Predicted in 506.79s
Resampling...
  Resampled in 0.66s
Saving segmentations...
Creating lung_airways.nii.gzCreating lung_arteries.nii.gzCreating lung_airways_wall.nii.gzCreating lung_veins.nii.gz



  Saved in 5.07s
Running VisionFeature segmentation
[model] vista3d_lpft_p2: Vista3DLPFTP2Trainer fold_0 final
[chkpt] /restricted/projectnb/batmanlab/haozhe/Lung_Mesh_Segment/data/nnUNetv2/nnUNet_results/Dataset502_ReXGvista3d/Vista3DLPFTP2Trainer__nnUNetResEncUNetLPlans_vista15__3d_fullres/fold_0/checkpoint_final.pth
[frame] RAS (nib.as_closest_canonical) -> transpose(2,1,0) = (Z,Y,X)
[frame] clip [-1000, 1000] -> [0,1] (baked; nnU-Net channel 'nonorm')
[frame] 1.5 mm isotropic (resample; logits interpolated back to native)
[backbone] VISTA3D loaded from /restricted/projectnb/batmanlab/haozhe/Lung_Mesh_Segment/VisionFeature/finetune_scripts/vista3d/checkpoints/vista3d/models--MONAI--vista3d/snapshots/c6dbe159632a4767696e09f91d74d729b82e73e6/models/model.pt


RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx